# Qwen3 Two-Stage LoRA SFT — General Evaluation → Abstract Evaluator

This notebook trains **one Qwen3 model with LoRA, not QLoRA**, in two stages:

1. **Stage 1: General evaluation SFT** on the EngSAF-style CSV files.
   - Train: `/home/MohammadNabulsi/Essay Evaluator/data/engsaf/clean/train/`
   - Validation: `/home/MohammadNabulsi/Essay Evaluator/data/engsaf/clean/validation/`
   - Test: `/home/MohammadNabulsi/Essay Evaluator/data/engsaf/clean/test/`
   - It automatically loads **every `.csv` file** inside each directory.

2. **Stage 2: Research-specific abstract evaluator SFT** on your own abstract-evaluation dataset.
   - It continues from the Stage 1 LoRA adapter.
   - It uses lower LR and 1–3 epochs.

The expected output format for both stages is always strict JSON:

```json
{"score": 2, "rationale": "..."}
```

## Main design choices

- **LoRA, not QLoRA**: `load_in_4bit=False`.
- **A100 80GB oriented**: bf16, gradient checkpointing, LoRA adapters only.
- **Two-stage curriculum**:
  - Stage 1 teaches general answer-evaluation behavior.
  - Stage 2 specializes the evaluator on research abstracts.
- **W&B/Hugging Face are optional** and controlled from one configuration cell.

## 0. Install dependencies

Run this once in a fresh environment. If your environment already has these packages, you can skip it.

In [ ]:
# %%capture
# !pip install -U transformers datasets trl accelerate peft sentencepiece protobuf safetensors
# !pip install -U unsloth
# !pip install -U wandb evaluate rouge_score sacrebleu bert_score scikit-learn pandas numpy tqdm huggingface_hub

## 1. Imports

In [ ]:
import os
import re
import gc
import ast
import json
import time
import random
import warnings
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch

from datasets import Dataset, DatasetDict
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, mean_absolute_error

from unsloth import FastLanguageModel
from transformers import TrainingArguments, EarlyStoppingCallback
from trl import SFTTrainer

import evaluate

warnings.filterwarnings("ignore")

SEED = 3407
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
# Force a stable W&B directory across sessions/devices
WANDB_DIR = Path("/home/MohammadNabulsi/Essay Evaluator/experiments/artifacts/wandb")
os.environ["WANDB_DIR"] = str(WANDB_DIR)
WANDB_DIR.mkdir(parents=True, exist_ok=True)


## 2. Global configuration

Edit only this cell for most runs.

In [ ]:
# =========================
# Paths
# =========================
PROJECT_ROOT = Path("/home/MohammadNabulsi/Essay Evaluator")

ENGSAF_TRAIN_DIR = PROJECT_ROOT / "data/engsaf/clean/train"
ENGSAF_VALIDATION_DIR = PROJECT_ROOT / "data/engsaf/clean/validation"
ENGSAF_TEST_DIR = PROJECT_ROOT / "data/engsaf/clean/test"

# Change this to your final research-specific abstract evaluator file.
# Supported: .csv, .parquet, .jsonl, .json
ABSTRACT_DATA_PATH = PROJECT_ROOT / "data/processed/final_abstract_eval_dataset.csv"

OUTPUT_ROOT = PROJECT_ROOT / "experiments/artifacts/qwen3_two_stage_lora_sft"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# =========================
# Model
# =========================
# Full LoRA, not QLoRA. Do NOT use a *bnb-4bit model here.
MODEL_NAME = "Qwen/Qwen3-8B"
MAX_SEQ_LENGTH = 2048
DTYPE = torch.bfloat16
LOAD_IN_4BIT = False

# =========================
# Optional logging / hub
# =========================
USE_WANDB = False
WANDB_PROJECT = "abstract-evaluator-two-stage-lora"
WANDB_ENTITY = None  # put your W&B entity/team here, or leave None

PUSH_TO_HUB = False
HF_REPO_ID = "Mohammad-Nabulsi/qwen3-abstract-evaluator-two-stage-lora"  # change if pushing

# =========================
# Stage 1: general evaluation SFT
# More LoRA layers unfrozen = lower freeze ratio.
# For a broad dataset, 1 epoch is usually enough as a behavior warm-up.
# =========================
GENERAL_RUN_NAME = "stage1_engsaf_general_eval_lora"
GENERAL_TRAIN_CFG = {
    "num_train_epochs": 1,
    "learning_rate": 1e-4,
    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.03,
    "weight_decay": 0.01,
    "logging_steps": 10,
    "eval_steps": 100,
    "save_steps": 100,
    "max_grad_norm": 0.3,
}
GENERAL_FREEZE_RATIO = 0.25  # freeze bottom 25% of LoRA layers, train top 75%

# =========================
# Stage 2: abstract-specific SFT
# Lower LR because it continues from stage 1.
# Use 1 epoch first; try 2 or 3 only if validation improves.
# =========================
ABSTRACT_RUN_NAME = "stage2_research_abstract_eval_lora"
ABSTRACT_TRAIN_CFG = {
    "num_train_epochs": 2,
    "learning_rate": 5e-5,
    "per_device_train_batch_size": 4,
    "per_device_eval_batch_size": 4,
    "gradient_accumulation_steps": 4,
    "warmup_ratio": 0.05,
    "weight_decay": 0.01,
    "logging_steps": 10,
    "eval_steps": 50,
    "save_steps": 50,
    "max_grad_norm": 0.3,
}
ABSTRACT_FREEZE_RATIO = 0.0  # train all LoRA layers, but with smaller LR

# =========================
# LoRA config
# =========================
LORA_CFG = {
    "r": 32,
    "lora_alpha": 64,
    "lora_dropout": 0.05,
    "bias": "none",
    "use_rslora": True,
    "target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
}

## 3. Optional W&B and Hugging Face login

Run this cell only if you want online logging or hub upload.

### W&B

Use one of these options:

```bash
export WANDB_API_KEY="your_key"
```

or run:

```python
import wandb
wandb.login()
```

### Hugging Face

Use one of these options:

```bash
export HF_TOKEN="your_token"
```

or run:

```python
from huggingface_hub import notebook_login
notebook_login()
```

In [ ]:
if USE_WANDB:
    import wandb
    if os.environ.get("WANDB_API_KEY"):
        wandb.login(key=os.environ["WANDB_API_KEY"])
    else:
        wandb.login()

if PUSH_TO_HUB:
    from huggingface_hub import login
    if os.environ.get("HF_TOKEN"):
        login(token=os.environ["HF_TOKEN"])
    else:
        from huggingface_hub import notebook_login
        notebook_login()

## 4. CSV loading helpers

In [ ]:
def read_all_csvs(directory: Path) -> pd.DataFrame:
    """Read and concatenate every CSV inside a directory."""
    paths = sorted(directory.glob("*.csv"))
    if not paths:
        raise FileNotFoundError(f"No .csv files found in: {directory}")

    frames = []
    for path in paths:
        df = pd.read_csv(path)
        df["source_file"] = path.name
        frames.append(df)

    out = pd.concat(frames, ignore_index=True)
    print(f"Loaded {len(paths)} CSV files from {directory}")
    print(f"Rows: {len(out):,}")
    return out


def load_file(path: Path) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(f"File does not exist: {path}")

    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pd.read_csv(path)
    if suffix in [".parquet", ".pq"]:
        return pd.read_parquet(path)
    if suffix == ".jsonl":
        return pd.read_json(path, lines=True)
    if suffix == ".json":
        return pd.read_json(path)
    raise ValueError(f"Unsupported file type: {suffix}")

## 5. Normalize EngSAF-style general evaluation data

Expected columns are similar to:

- `question`
- `student_answer`
- `reference_answer`
- `mark_scheme`
- `score`
- `rationale`

The notebook will fail loudly if any required column is missing.

In [ ]:
ENGSAF_REQUIRED = ["question", "student_answer", "reference_answer", "mark_scheme", "score", "rationale"]


def parse_possible_dict(x, default=None):
    if isinstance(x, dict):
        return {str(k): str(v) for k, v in x.items()}
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return default
    if isinstance(x, str):
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, dict):
                return {str(k): str(v) for k, v in parsed.items()}
        except Exception:
            pass
    return default


def clean_score(x) -> int:
    if pd.isna(x):
        raise ValueError("Score contains NaN")
    return int(float(x))


def normalize_engsaf_df(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    missing = [c for c in ENGSAF_REQUIRED if c not in df.columns]
    if missing:
        raise ValueError(f"EngSAF {split_name} missing columns: {missing}")

    out = df.copy()
    out = out.dropna(subset=["question", "student_answer", "reference_answer", "score", "rationale"]).reset_index(drop=True)

    out["score"] = out["score"].apply(clean_score)
    out["rubric"] = out["mark_scheme"].apply(
        lambda x: parse_possible_dict(x, default={"0": "Incorrect response", "1": "Partially correct response", "2": "Correct response"})
    )

    normalized = pd.DataFrame({
        "id": [f"engsaf_{split_name}_{i}" for i in range(len(out))],
        "dataset_type": "general_evaluation",
        "split": split_name,
        "source_file": out.get("source_file", pd.Series([None] * len(out))),
        "task": "Grade the student answer using the question, reference answer, and mark scheme.",
        "question": out["question"].astype(str),
        "student_answer": out["student_answer"].astype(str),
        "reference_answer": out["reference_answer"].astype(str),
        "rubric": out["rubric"],
        "score": out["score"].astype(int),
        "rationale": out["rationale"].astype(str),
    })

    return normalized

## 6. Load EngSAF train/validation/test splits

In [ ]:
engsaf_train_raw = read_all_csvs(ENGSAF_TRAIN_DIR)
engsaf_val_raw = read_all_csvs(ENGSAF_VALIDATION_DIR)
engsaf_test_raw = read_all_csvs(ENGSAF_TEST_DIR)

engsaf_train_df = normalize_engsaf_df(engsaf_train_raw, "train")
engsaf_val_df = normalize_engsaf_df(engsaf_val_raw, "validation")
engsaf_test_df = normalize_engsaf_df(engsaf_test_raw, "test")

for name, part in [("train", engsaf_train_df), ("validation", engsaf_val_df), ("test", engsaf_test_df)]:
    print(name, part.shape)
    print(part["score"].value_counts(normalize=True).sort_index())

engsaf_train_df.head()

## 7. Normalize your abstract-evaluation dataset

Expected minimum columns:

- `submission` or `abstract`
- `score`
- `rationale`

Recommended columns:

- `paper_id`
- `title`
- `rubric`
- `reference`
- `task`

If no validation/test files are provided for the abstract dataset, this notebook splits by `paper_id` when available to reduce leakage.

In [ ]:
DEFAULT_ABSTRACT_TASK = "Evaluate the quality of the following research abstract for conference acceptance."
DEFAULT_ABSTRACT_REFERENCE = "A strong research abstract clearly presents the problem, objective, methodology, contribution, and evidence/results."
DEFAULT_ABSTRACT_RUBRIC = {
    "0": "Very weak / unacceptable abstract",
    "1": "Weak abstract with major missing components",
    "2": "Borderline abstract with partial coverage",
    "3": "Good abstract with minor issues",
    "4": "Strong abstract with clear problem, method, contribution, and evidence",
}


def first_existing_column(df: pd.DataFrame, candidates: List[str]) -> Optional[str]:
    for c in candidates:
        if c in df.columns:
            return c
    return None


def normalize_abstract_df(df: pd.DataFrame) -> pd.DataFrame:
    submission_col = first_existing_column(df, ["submission", "abstract", "student_answer", "answer", "text"])
    if submission_col is None:
        raise ValueError("Abstract dataset needs one of: submission, abstract, student_answer, answer, text")
    if "score" not in df.columns or "rationale" not in df.columns:
        raise ValueError("Abstract dataset must include score and rationale columns")

    out = df.copy().dropna(subset=[submission_col, "score", "rationale"]).reset_index(drop=True)
    out["score"] = out["score"].apply(clean_score)

    paper_id = out["paper_id"].astype(str) if "paper_id" in out.columns else pd.Series([f"abstract_{i}" for i in range(len(out))])
    title = out["title"].astype(str) if "title" in out.columns else pd.Series([""] * len(out))
    task = out["task"].astype(str) if "task" in out.columns else pd.Series([DEFAULT_ABSTRACT_TASK] * len(out))
    reference = out["reference"].astype(str) if "reference" in out.columns else pd.Series([DEFAULT_ABSTRACT_REFERENCE] * len(out))

    if "rubric" in out.columns:
        rubric = out["rubric"].apply(lambda x: parse_possible_dict(x, default=DEFAULT_ABSTRACT_RUBRIC))
    else:
        rubric = pd.Series([DEFAULT_ABSTRACT_RUBRIC] * len(out))

    normalized = pd.DataFrame({
        "id": [f"abstract_{i}" for i in range(len(out))],
        "paper_id": paper_id,
        "dataset_type": "research_abstract_evaluation",
        "task": task,
        "title": title,
        "submission": out[submission_col].astype(str),
        "reference": reference,
        "rubric": rubric,
        "score": out["score"].astype(int),
        "rationale": out["rationale"].astype(str),
    })
    return normalized


def split_abstract_dataset(df: pd.DataFrame, seed: int = SEED) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    if "paper_id" in df.columns:
        ids = df["paper_id"].astype(str).unique()
        train_ids, temp_ids = train_test_split(ids, train_size=0.80, random_state=seed, shuffle=True)
        val_ids, test_ids = train_test_split(temp_ids, test_size=0.50, random_state=seed, shuffle=True)
        train = df[df["paper_id"].astype(str).isin(train_ids)].reset_index(drop=True)
        val = df[df["paper_id"].astype(str).isin(val_ids)].reset_index(drop=True)
        test = df[df["paper_id"].astype(str).isin(test_ids)].reset_index(drop=True)
    else:
        train, temp = train_test_split(df, train_size=0.80, random_state=seed, shuffle=True, stratify=df["score"] if df["score"].nunique() > 1 else None)
        val, test = train_test_split(temp, test_size=0.50, random_state=seed, shuffle=True, stratify=temp["score"] if temp["score"].nunique() > 1 else None)
        train, val, test = train.reset_index(drop=True), val.reset_index(drop=True), test.reset_index(drop=True)
    return train, val, test

## 8. Load abstract dataset

Update `ABSTRACT_DATA_PATH` in the config cell if this path is different.

In [ ]:
abstract_raw = load_file(ABSTRACT_DATA_PATH)
abstract_df = normalize_abstract_df(abstract_raw)
abstract_train_df, abstract_val_df, abstract_test_df = split_abstract_dataset(abstract_df)

for name, part in [("train", abstract_train_df), ("validation", abstract_val_df), ("test", abstract_test_df)]:
    print(name, part.shape)
    print(part["score"].value_counts(normalize=True).sort_index())

abstract_train_df.head()

## 9. Prompt/message construction

Both stages use the same JSON output contract, but the input format is adapted to the dataset type.

In [ ]:
def format_rubric(rubric: Dict[str, str]) -> str:
    if not isinstance(rubric, dict):
        rubric = parse_possible_dict(rubric, default={}) or {}
    def sort_key(item):
        k = str(item[0])
        try:
            return int(k)
        except Exception:
            return k
    return "\n".join([f"{k} = {v}" for k, v in sorted(rubric.items(), key=sort_key)])


def make_general_eval_user_prompt(row: pd.Series) -> str:
    return (
        "/no_think\n"
        "Task:\n"
        f"{row['task']}\n\n"
        "Question:\n"
        f"{row['question']}\n\n"
        "Student answer:\n"
        f"{row['student_answer']}\n\n"
        "Reference answer:\n"
        f"{row['reference_answer']}\n\n"
        "Mark scheme:\n"
        f"{format_rubric(row['rubric'])}\n\n"
        "Return only valid JSON with exactly these keys: score, rationale.\n"
        "Do not include markdown, analysis, or extra text."
    )


def make_abstract_eval_user_prompt(row: pd.Series) -> str:
    title_block = ""
    if "title" in row and pd.notna(row["title"]) and str(row["title"]).strip():
        title_block = f"Title:\n{str(row['title']).strip()}\n\n"
    return (
        "/no_think\n"
        "Task:\n"
        f"{row['task']}\n\n"
        "Reference standard:\n"
        f"{row['reference']}\n\n"
        "Rubric:\n"
        f"{format_rubric(row['rubric'])}\n\n"
        f"{title_block}"
        "Abstract/submission:\n"
        f"{row['submission']}\n\n"
        "Return only valid JSON with exactly these keys: score, rationale.\n"
        "Do not include markdown, analysis, or extra text."
    )


def make_user_prompt(row: pd.Series) -> str:
    if row["dataset_type"] == "general_evaluation":
        return make_general_eval_user_prompt(row)
    if row["dataset_type"] == "research_abstract_evaluation":
        return make_abstract_eval_user_prompt(row)
    raise ValueError(f"Unknown dataset_type: {row['dataset_type']}")


def make_assistant_answer(row: pd.Series) -> str:
    return json.dumps({"score": int(row["score"]), "rationale": str(row["rationale"]).strip()}, ensure_ascii=False)


def make_messages(row: pd.Series) -> List[Dict[str, str]]:
    return [
        {
            "role": "system",
            "content": "You are a strict educational and research evaluator. Return only valid JSON.",
        },
        {"role": "user", "content": make_user_prompt(row)},
        {"role": "assistant", "content": make_assistant_answer(row)},
    ]


def attach_messages(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["target_json"] = out.apply(make_assistant_answer, axis=1)
    out["messages"] = out.apply(make_messages, axis=1)
    return out

engsaf_train_df = attach_messages(engsaf_train_df)
engsaf_val_df = attach_messages(engsaf_val_df)
engsaf_test_df = attach_messages(engsaf_test_df)

abstract_train_df = attach_messages(abstract_train_df)
abstract_val_df = attach_messages(abstract_val_df)
abstract_test_df = attach_messages(abstract_test_df)

print(json.dumps(engsaf_train_df.iloc[0]["messages"], indent=2, ensure_ascii=False)[:2000])

## 10. Convert DataFrames to Hugging Face datasets

In [ ]:
def to_hf_dataset(df_part: pd.DataFrame) -> Dataset:
    keep_cols = [
        "id", "dataset_type", "messages", "score", "rationale", "target_json",
        "paper_id", "source_file",
    ]
    keep_cols = [c for c in keep_cols if c in df_part.columns]
    return Dataset.from_pandas(df_part[keep_cols], preserve_index=False)

engsaf_ds = DatasetDict({
    "train": to_hf_dataset(engsaf_train_df),
    "validation": to_hf_dataset(engsaf_val_df),
    "test": to_hf_dataset(engsaf_test_df),
})

abstract_ds = DatasetDict({
    "train": to_hf_dataset(abstract_train_df),
    "validation": to_hf_dataset(abstract_val_df),
    "test": to_hf_dataset(abstract_test_df),
})

engsaf_ds, abstract_ds

## 11. Qwen3 chat template helpers

The function tries to use `enable_thinking=False`. If your tokenizer version does not support it, it falls back safely.

In [ ]:
def apply_qwen3_chat_template(tokenizer, messages, add_generation_prompt=False):
    try:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
            enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=add_generation_prompt,
        )


def build_formatting_func(tokenizer):
    def formatting_prompts_func(examples):
        texts = []
        for messages in examples["messages"]:
            text = apply_qwen3_chat_template(tokenizer, messages, add_generation_prompt=False)
            if tokenizer.eos_token and not text.endswith(tokenizer.eos_token):
                text += tokenizer.eos_token
            texts.append(text)
        return texts
    return formatting_prompts_func

## 12. Load Qwen3 with LoRA, not QLoRA

Important: this cell uses `load_in_4bit=False`. That is what makes this **LoRA** rather than **QLoRA**.

In [ ]:
def infer_num_layers(model) -> int:
    cfg = model.config
    for attr in ["num_hidden_layers", "n_layers", "num_layers"]:
        if hasattr(cfg, attr):
            return int(getattr(cfg, attr))
    raise ValueError("Could not infer transformer layer count.")


def set_lora_layer_freeze_ratio(model, freeze_ratio: float):
    """Freeze LoRA params in lower transformer layers, keep upper LoRA params trainable."""
    freeze_ratio = float(freeze_ratio)
    num_layers = infer_num_layers(model)
    cutoff = int(num_layers * freeze_ratio)
    layer_pat = re.compile(r"\.layers\.(\d+)\.")

    trainable = 0
    frozen = 0

    for name, param in model.named_parameters():
        # Base model should remain frozen in PEFT/LoRA. Only LoRA params are candidates.
        if "lora_" not in name.lower():
            param.requires_grad = False
            frozen += param.numel()
            continue

        match = layer_pat.search(name)
        if match and int(match.group(1)) < cutoff:
            param.requires_grad = False
            frozen += param.numel()
        else:
            param.requires_grad = True
            trainable += param.numel()

    info = {
        "num_layers": num_layers,
        "freeze_ratio": freeze_ratio,
        "frozen_lora_below_layer": cutoff,
        "trainable_params": trainable,
        "frozen_params": frozen,
    }
    print(json.dumps(info, indent=2))
    return info


def load_model_and_tokenizer():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=DTYPE,
        load_in_4bit=LOAD_IN_4BIT,
    )

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    model = FastLanguageModel.get_peft_model(
        model,
        r=LORA_CFG["r"],
        target_modules=LORA_CFG["target_modules"],
        lora_alpha=LORA_CFG["lora_alpha"],
        lora_dropout=LORA_CFG["lora_dropout"],
        bias=LORA_CFG["bias"],
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        use_rslora=LORA_CFG["use_rslora"],
    )
    return model, tokenizer

model, tokenizer = load_model_and_tokenizer()
formatting_func = build_formatting_func(tokenizer)

## 13. Trainer builder

In [ ]:
def make_training_args(run_name: str, cfg: Dict[str, Any], output_dir: Path) -> TrainingArguments:
    return TrainingArguments(
        output_dir=str(output_dir),
        overwrite_output_dir=True,
        run_name=run_name,

        num_train_epochs=cfg["num_train_epochs"],
        per_device_train_batch_size=cfg["per_device_train_batch_size"],
        per_device_eval_batch_size=cfg["per_device_eval_batch_size"],
        gradient_accumulation_steps=cfg["gradient_accumulation_steps"],

        learning_rate=cfg["learning_rate"],
        warmup_ratio=cfg["warmup_ratio"],
        weight_decay=cfg["weight_decay"],
        max_grad_norm=cfg["max_grad_norm"],
        lr_scheduler_type="cosine",

        bf16=True,
        fp16=False,
        optim="adamw_torch",

        logging_steps=cfg["logging_steps"],
        eval_strategy="steps",
        eval_steps=cfg["eval_steps"],
        save_strategy="steps",
        save_steps=cfg["save_steps"],
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,

        report_to="wandb" if USE_WANDB else "none",
        push_to_hub=False,
        remove_unused_columns=False,
        seed=SEED,
    )


def make_sft_trainer(model, tokenizer, train_dataset, eval_dataset, args):
    kwargs = dict(
        model=model,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        formatting_func=formatting_func,
        args=args,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    )

    # TRL changed tokenizer -> processing_class in newer versions.
    try:
        return SFTTrainer(**kwargs, processing_class=tokenizer)
    except TypeError:
        return SFTTrainer(**kwargs, tokenizer=tokenizer)

## 14. Stage 1 training — general EngSAF evaluation

This trains for one epoch on all CSVs in the EngSAF training folder and validates on all CSVs in the validation folder.

In [ ]:
if USE_WANDB:
    import wandb
    wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=GENERAL_RUN_NAME,
        config={
            "model_name": MODEL_NAME,
            "stage": "general_evaluation",
            "load_in_4bit": LOAD_IN_4BIT,
            "max_seq_length": MAX_SEQ_LENGTH,
            "freeze_ratio": GENERAL_FREEZE_RATIO,
            **GENERAL_TRAIN_CFG,
            **{f"lora_{k}": v for k, v in LORA_CFG.items() if k != "target_modules"},
            "target_modules": LORA_CFG["target_modules"],
        },
        reinit=True,
    )

set_lora_layer_freeze_ratio(model, GENERAL_FREEZE_RATIO)

general_output_dir = OUTPUT_ROOT / "models" / GENERAL_RUN_NAME
general_args = make_training_args(GENERAL_RUN_NAME, GENERAL_TRAIN_CFG, general_output_dir)

general_trainer = make_sft_trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=engsaf_ds["train"],
    eval_dataset=engsaf_ds["validation"],
    args=general_args,
)

general_train_result = general_trainer.train()
general_trainer.save_model(str(general_output_dir / "final_adapter"))
tokenizer.save_pretrained(str(general_output_dir / "final_adapter"))

if USE_WANDB:
    wandb.finish()

print("Saved Stage 1 adapter to:", general_output_dir / "final_adapter")

## 15. Stage 2 training — research-specific abstract evaluator

This continues from the same model after Stage 1. It uses lower learning rate and trains the research-specific task for 1–3 epochs.

In [ ]:
# Free memory that is not needed from the previous Trainer object.
del general_trainer
gc.collect()
torch.cuda.empty_cache()

if USE_WANDB:
    import wandb
    wandb.init(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        name=ABSTRACT_RUN_NAME,
        config={
            "model_name": MODEL_NAME,
            "stage": "research_abstract_evaluation",
            "starts_from": str(general_output_dir / "final_adapter"),
            "load_in_4bit": LOAD_IN_4BIT,
            "max_seq_length": MAX_SEQ_LENGTH,
            "freeze_ratio": ABSTRACT_FREEZE_RATIO,
            **ABSTRACT_TRAIN_CFG,
            **{f"lora_{k}": v for k, v in LORA_CFG.items() if k != "target_modules"},
            "target_modules": LORA_CFG["target_modules"],
        },
        reinit=True,
    )

set_lora_layer_freeze_ratio(model, ABSTRACT_FREEZE_RATIO)

abstract_output_dir = OUTPUT_ROOT / "models" / ABSTRACT_RUN_NAME
abstract_args = make_training_args(ABSTRACT_RUN_NAME, ABSTRACT_TRAIN_CFG, abstract_output_dir)

abstract_trainer = make_sft_trainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=abstract_ds["train"],
    eval_dataset=abstract_ds["validation"],
    args=abstract_args,
)

abstract_train_result = abstract_trainer.train()
abstract_trainer.save_model(str(abstract_output_dir / "final_adapter"))
tokenizer.save_pretrained(str(abstract_output_dir / "final_adapter"))

if USE_WANDB:
    wandb.finish()

print("Saved Stage 2 adapter to:", abstract_output_dir / "final_adapter")

## 16. Optional: push final LoRA adapter to Hugging Face

In [ ]:
if PUSH_TO_HUB:
    abstract_trainer.push_to_hub(repo_id=HF_REPO_ID)
    tokenizer.push_to_hub(HF_REPO_ID)
    print("Pushed to:", HF_REPO_ID)
else:
    print("PUSH_TO_HUB=False, skipping upload.")

## 17. Generation and JSON parsing helpers

In [ ]:
def extract_json_object(text: str) -> Optional[Dict[str, Any]]:
    if not isinstance(text, str):
        return None
    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            return obj
    except Exception:
        pass
    match = re.search(r"\{.*?\}", str(text), flags=re.S)
    if match:
        try:
            obj = json.loads(match.group(0))
            if isinstance(obj, dict):
                return obj
        except Exception:
            return None
    return None


def parse_score(text: str) -> Optional[int]:
    obj = extract_json_object(text)
    if obj is not None and "score" in obj:
        try:
            return int(obj["score"])
        except Exception:
            pass
    match = re.search(r'"?score"?\s*[:=]\s*(-?\d+)', str(text))
    if match:
        return int(match.group(1))
    return None


def parse_rationale(text: str) -> str:
    obj = extract_json_object(text)
    if obj is not None and "rationale" in obj:
        return str(obj["rationale"]).strip()
    return str(text).strip()


def make_inference_messages(messages: List[Dict[str, str]]) -> List[Dict[str, str]]:
    return [m for m in messages if m["role"] in ["system", "user"]]


def generate_predictions(df: pd.DataFrame, max_new_tokens: int = 180, batch_size: int = 4) -> pd.DataFrame:
    FastLanguageModel.for_inference(model)
    rows = []

    prompts = []
    for _, row in df.iterrows():
        inference_messages = make_inference_messages(row["messages"])
        prompt = apply_qwen3_chat_template(tokenizer, inference_messages, add_generation_prompt=True)
        prompts.append(prompt)

    for start in range(0, len(prompts), batch_size):
        batch_prompts = prompts[start:start + batch_size]
        inputs = tokenizer(
            batch_prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=None,
                top_p=None,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        for i, output_ids in enumerate(outputs):
            prompt_len = inputs["input_ids"][i].shape[0]
            gen_ids = output_ids[prompt_len:]
            prediction_text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

            original_row = df.iloc[start + i].to_dict()
            original_row["prediction_text"] = prediction_text
            original_row["pred_score"] = parse_score(prediction_text)
            original_row["pred_rationale"] = parse_rationale(prediction_text)
            rows.append(original_row)

    return pd.DataFrame(rows)

## 18. Metrics

Score metrics are usually more important than ROUGE/BLEU for this task. Text metrics are included only as rough diagnostics for rationale style.

In [ ]:
rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("sacrebleu")

try:
    bertscore_metric = evaluate.load("bertscore")
except Exception:
    bertscore_metric = None


def compute_eval_metrics(pred_df: pd.DataFrame, prefix: str) -> Dict[str, float]:
    out = {}
    valid_mask = pred_df["pred_score"].notna()
    out[f"{prefix}/json_parse_rate"] = float(valid_mask.mean())

    if valid_mask.any():
        y_true = pred_df.loc[valid_mask, "score"].astype(int).values
        y_pred = pred_df.loc[valid_mask, "pred_score"].astype(int).values
        out[f"{prefix}/score_accuracy"] = float(accuracy_score(y_true, y_pred))
        out[f"{prefix}/score_mae"] = float(mean_absolute_error(y_true, y_pred))
        out[f"{prefix}/score_within_1_accuracy"] = float((np.abs(y_true - y_pred) <= 1).mean())
    else:
        out[f"{prefix}/score_accuracy"] = 0.0
        out[f"{prefix}/score_mae"] = 999.0
        out[f"{prefix}/score_within_1_accuracy"] = 0.0

    preds = pred_df["pred_rationale"].fillna("").astype(str).tolist()
    refs = pred_df["rationale"].fillna("").astype(str).tolist()

    try:
        rouge = rouge_metric.compute(predictions=preds, references=refs)
        out.update({f"{prefix}/rouge_{k}": float(v) for k, v in rouge.items()})
    except Exception as e:
        print("ROUGE failed:", e)

    try:
        bleu = bleu_metric.compute(predictions=preds, references=[[r] for r in refs])
        out[f"{prefix}/bleu"] = float(bleu["score"])
    except Exception as e:
        print("BLEU failed:", e)

    if bertscore_metric is not None:
        try:
            bs = bertscore_metric.compute(predictions=preds, references=refs, lang="en")
            out[f"{prefix}/bertscore_f1_mean"] = float(np.mean(bs["f1"]))
        except Exception as e:
            print("BERTScore failed:", e)

    return out

## 19. Final evaluation on EngSAF test and abstract test

After Stage 2, the model is specialized for abstracts, but EngSAF test performance is still useful to check whether it forgot general evaluation behavior.

In [ ]:
eval_out_dir = OUTPUT_ROOT / "eval" / ABSTRACT_RUN_NAME
eval_out_dir.mkdir(parents=True, exist_ok=True)

print("Generating EngSAF test predictions...")
engsaf_test_pred = generate_predictions(
    engsaf_test_df,
    max_new_tokens=180,
    batch_size=ABSTRACT_TRAIN_CFG["per_device_eval_batch_size"],
)
engsaf_test_pred.to_csv(eval_out_dir / "engsaf_test_predictions.csv", index=False)
engsaf_metrics = compute_eval_metrics(engsaf_test_pred, "engsaf_test")

print("Generating abstract validation predictions...")
abstract_val_pred = generate_predictions(
    abstract_val_df,
    max_new_tokens=180,
    batch_size=ABSTRACT_TRAIN_CFG["per_device_eval_batch_size"],
)
abstract_val_pred.to_csv(eval_out_dir / "abstract_validation_predictions.csv", index=False)
abstract_val_metrics = compute_eval_metrics(abstract_val_pred, "abstract_validation")

print("Generating abstract test predictions...")
abstract_test_pred = generate_predictions(
    abstract_test_df,
    max_new_tokens=180,
    batch_size=ABSTRACT_TRAIN_CFG["per_device_eval_batch_size"],
)
abstract_test_pred.to_csv(eval_out_dir / "abstract_test_predictions.csv", index=False)
abstract_test_metrics = compute_eval_metrics(abstract_test_pred, "abstract_test")

metrics = {
    "model_name": MODEL_NAME,
    "load_in_4bit": LOAD_IN_4BIT,
    "stage1_run": GENERAL_RUN_NAME,
    "stage2_run": ABSTRACT_RUN_NAME,
    **engsaf_metrics,
    **abstract_val_metrics,
    **abstract_test_metrics,
}

with open(eval_out_dir / "metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2, ensure_ascii=False)

if USE_WANDB:
    import wandb
    wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=f"{ABSTRACT_RUN_NAME}_final_eval", reinit=True)
    wandb.log(metrics)
    wandb.finish()

print(json.dumps(metrics, indent=2))
print("Saved evaluation outputs to:", eval_out_dir)

## 20. Inspect worst abstract scoring errors

In [ ]:
def add_error_columns(pred_df: pd.DataFrame) -> pd.DataFrame:
    out = pred_df.copy()
    out["pred_score_numeric"] = pd.to_numeric(out["pred_score"], errors="coerce")
    out["score_error"] = out["pred_score_numeric"] - out["score"]
    out["abs_score_error"] = out["score_error"].abs()
    return out

inspected = add_error_columns(abstract_test_pred)

cols = [c for c in [
    "id", "paper_id", "score", "pred_score", "rationale", "pred_rationale", "prediction_text"
] if c in inspected.columns]

inspected.sort_values("abs_score_error", ascending=False).head(20)[cols]

## 21. How to tune this notebook

### Good default

Keep:

```python
GENERAL_TRAIN_CFG["num_train_epochs"] = 1
ABSTRACT_TRAIN_CFG["num_train_epochs"] = 2
GENERAL_TRAIN_CFG["learning_rate"] = 1e-4
ABSTRACT_TRAIN_CFG["learning_rate"] = 5e-5
GENERAL_FREEZE_RATIO = 0.25
ABSTRACT_FREEZE_RATIO = 0.0
```

### If abstract validation loss increases early

Use:

```python
ABSTRACT_TRAIN_CFG["num_train_epochs"] = 1
ABSTRACT_TRAIN_CFG["learning_rate"] = 2e-5
ABSTRACT_FREEZE_RATIO = 0.25
```

### If the model underfits

Use:

```python
LORA_CFG["r"] = 64
LORA_CFG["lora_alpha"] = 128
ABSTRACT_TRAIN_CFG["num_train_epochs"] = 3
ABSTRACT_FREEZE_RATIO = 0.0
```

### If you get OOM on A100 80GB

Use:

```python
GENERAL_TRAIN_CFG["per_device_train_batch_size"] = 2
ABSTRACT_TRAIN_CFG["per_device_train_batch_size"] = 2
GENERAL_TRAIN_CFG["gradient_accumulation_steps"] = 8
ABSTRACT_TRAIN_CFG["gradient_accumulation_steps"] = 8
```

### Important note

This notebook intentionally avoids QLoRA. If memory becomes a problem and you switch to `load_in_4bit=True`, then it becomes QLoRA again.